In [1]:
import pandas as pd
import numpy as np

In [2]:
tags_table = pd.read_excel('taglist.xlsx')
tags_table.head(2)

,name,desc,scale_min,scale_max,unit,dec,lolo,lo,hi,hihi,type
0,2504-FA,Пожар в объекте 2504,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,discret
1,412/6,Электрообогрев 412/6 отключить,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,app


In [3]:
columns = ["scale_min","scale_max","lolo","lo","hi","hihi"]
for colmn in columns:
    tags_table[colmn] = tags_table[colmn].astype(str).str.replace(",", ".", regex=False)
    tags_table[colmn] = pd.to_numeric(tags_table[colmn], errors="coerce")#lkz случаев с нан или -

In [4]:
tags_table = tags_table.astype({'name':'object','desc':'object','unit':'object',
                                    'scale_min':'float64','scale_max':'float64','dec':'Int64','lo':'float64','lolo':'float64',
                                'hi':'float64','hihi':'float64', 'type':'object'})

In [5]:
tags_table.dtypes

name          object
desc          object
scale_min    float64
scale_max    float64
unit          object
dec            Int64
lolo         float64
lo           float64
hi           float64
hihi         float64
type          object
dtype: object

In [6]:
columns_template = ['name','tag_type','initial_value','desc','unit','scale_min','scale_max',
            'dec','access_type','group_name','logged','lo','lolo','hi','hihi','discrete_alarm_value']
table_result = tags_table.reindex(columns=columns_template + [c for c in tags_table.columns if c not in columns_template])
table_result.head(2)

,name,tag_type,initial_value,desc,unit,scale_min,scale_max,dec,access_type,group_name,logged,lo,lolo,hi,hihi,discrete_alarm_value,type
0,2504-FA,NaN,NaN,Пожар в объекте 2504,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,discret
1,412/6,NaN,NaN,Электрообогрев 412/6 отключить,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,app


In [7]:
table_result = table_result.astype({'name':'object','tag_type':'Int64','initial_value':'Int64','desc':'object','unit':'object',
                                    'scale_min':'float64','scale_max':'float64','dec':'Int64','access_type':'Int64',
                                    'group_name':'Int64','logged':'Int64','lo':'float64','lolo':'float64','hi':'float64',
                                    'hihi':'float64','discrete_alarm_value':'Int64'})
table_result.dtypes

name                     object
tag_type                  Int64
initial_value             Int64
desc                     object
unit                     object
scale_min               float64
scale_max               float64
dec                       Int64
access_type               Int64
group_name                Int64
logged                    Int64
lo                      float64
lolo                    float64
hi                      float64
hihi                    float64
discrete_alarm_value      Int64
type                     object
dtype: object

**Датчики**  
- поломки  
**Регуляторы**  
- мв
 - св
- мод
- поломки1
- поломки2
**Сделать структуру таблицы**

In [8]:
# Проверка на русские буквы
table_result["name_copy"] = table_result["name"]
tags_rename = {
   'а':'a','б':'b','в':'v','г':'g','д':'d','е':'e','ё':'yo','ж':'zh','з':'z','и':'i','й':'y',
    'к':'k','л':'l','м':'m','н':'n','о':'o','п':'p','р':'r','с':'s','т':'t','у':'u','ф':'f',
    'х':'kh','ц':'ts','ч':'ch','ш':'sh','щ':'shch','ъ':'','ы':'y','ь':'','э':'e','ю':'yu','я':'ya',
    'А':'A','Б':'B','В':'V','Г':'G','Д':'D','Е':'E','Ё':'Yo','Ж':'Zh','З':'Z','И':'I','Й':'Y',
    'К':'K','Л':'L','М':'M','Н':'N','О':'O','П':'P','Р':'R','С':'S','Т':'T','У':'U','Ф':'F',
    'Х':'Kh','Ц':'Ts','Ч':'Ch','Ш':'Sh','Щ':'Shch','Ъ':'','Ы':'Y','Ь':'','Э':'E','Ю':'Yu','Я':'Ya',
    '\\':'_','-':'_',' ':'','/':'_'}
table_result['name'] = (table_result['name'].astype(str).str.translate(str.maketrans(tags_rename)))
table_result.head(2)

,name,tag_type,initial_value,desc,unit,scale_min,scale_max,dec,access_type,group_name,logged,lo,lolo,hi,hihi,discrete_alarm_value,type,name_copy
0,2504_FA,<NA>,<NA>,Пожар в объекте 2504,NaN,NaN,NaN,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,<NA>,discret,2504-FA
1,412_6,<NA>,<NA>,Электрообогрев 412/6 отключить,NaN,NaN,NaN,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,<NA>,app,412/6


In [9]:
# Проверка что начинается с цифры
prefix = 'K_'
#tags_table.loc[tags_table["name"].str.match(r"^\d"), "name"] = prefix + tags_table["name"]

In [10]:
def add_prefix(x):
    x = str(x)
    if x[0].isdigit():
        return prefix + x
    else:
        return x
table_result["name"] = table_result["name"].apply(add_prefix)
table_result.head(2)

,name,tag_type,initial_value,desc,unit,scale_min,scale_max,dec,access_type,group_name,logged,lo,lolo,hi,hihi,discrete_alarm_value,type,name_copy
0,K_2504_FA,<NA>,<NA>,Пожар в объекте 2504,NaN,NaN,NaN,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,<NA>,discret,2504-FA
1,K_412_6,<NA>,<NA>,Электрообогрев 412/6 отключить,NaN,NaN,NaN,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,<NA>,app,412/6


In [11]:
# Добавить всем _PV
def add_pv(x):
    x = str(x)
    if not x.endswith("_PV"):
        return x + '_PV'
    else:
        return x
table_result["name"] = table_result["name"].apply(add_pv)
table_result.head(2)

,name,tag_type,initial_value,desc,unit,scale_min,scale_max,dec,access_type,group_name,logged,lo,lolo,hi,hihi,discrete_alarm_value,type,name_copy
0,K_2504_FA_PV,<NA>,<NA>,Пожар в объекте 2504,NaN,NaN,NaN,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,<NA>,discret,2504-FA
1,K_412_6_PV,<NA>,<NA>,Электрообогрев 412/6 отключить,NaN,NaN,NaN,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,<NA>,app,412/6


In [12]:
# Расстановка необходимых значений для каждого типа тега
table_result['initial_value'] = 0 # здесь всегда 0, значение при запуске
table_result['access_type'] = 1 # здесь всегда 1, тип доступа


In [13]:
#tag_type
type_1 = table_result["type"] != "discret"
table_result.loc[type_1, "tag_type"] = 1 # type analog
type_3 = table_result["type"] == "discret"
table_result.loc[type_3, "tag_type"] = 3 #type discret
# fire discret alarm
table_result.loc[type_3, "discrete_alarm_value"] = 1 #fire alarme discret

In [14]:
type_app = table_result["type"] == "app"
table_result.loc[type_app, "scale_min"] = table_result["scale_min"].fillna(0)
table_result.loc[type_app, "scale_max"] = table_result["scale_max"].fillna(1)
#table_result["scale_min"] = table_result["scale_min"].fillna(0)
#table_result["scale_max"] = table_result["scale_max"].fillna(1)

In [15]:
# logging
type_logging = table_result["type"].isin(['reg','analog'])
table_result.loc[type_logging, "logged"] = 1
table_result["logged"] = table_result["logged"].fillna(0)

In [16]:
# units
unit_rename = {
    "нм3/час": "нм3/ч",
    "кг/час" : "кг/ч",
    "кПа" : "КПа",
    "Мпа" : "МПа",
    "С": "°C",
    "C" : "°C",
    "м3/час": "м3/ч",
}
table_result['unit'] = table_result['unit'].replace(unit_rename)
table_result.head(2)

,name,tag_type,initial_value,desc,unit,scale_min,scale_max,dec,access_type,group_name,logged,lo,lolo,hi,hihi,discrete_alarm_value,type,name_copy
0,K_2504_FA_PV,3,0,Пожар в объекте 2504,NaN,NaN,NaN,<NA>,1,<NA>,0,NaN,NaN,NaN,NaN,1,discret,2504-FA
1,K_412_6_PV,1,0,Электрообогрев 412/6 отключить,NaN,0.0,1.0,<NA>,1,<NA>,0,NaN,NaN,NaN,NaN,<NA>,app,412/6


In [17]:
##Поломки для датчиков
br_analog = table_result[table_result["type"] == 'analog'].copy()
br_analog['name'] = br_analog['name'].astype(str).str.replace("_PV", "_BR", regex=False)
br_analog['desc'] = 'Поломки'
br_analog['scale_min'] = 0
br_analog['scale_max'] = 6
br_analog['dec'] = np.nan
br_analog['lolo'] = np.nan
br_analog['lo'] = np.nan
br_analog['hi'] = np.nan
br_analog['hihi'] = np.nan
br_analog['unit'] = np.nan
br_analog['logged'] = 0
br_analog.head(2)

,name,tag_type,initial_value,desc,unit,scale_min,scale_max,dec,access_type,group_name,logged,lo,lolo,hi,hihi,discrete_alarm_value,type,name_copy
46,FI3023_BR,1,0,Поломки,NaN,0,6,NaN,1,<NA>,0,NaN,NaN,NaN,NaN,<NA>,analog,FI 3023
47,FI3372_BR,1,0,Поломки,NaN,0,6,NaN,1,<NA>,0,NaN,NaN,NaN,NaN,<NA>,analog,FI 3372


In [18]:
##Поломки для регуляторов
br_reg = table_result[table_result["type"] == 'reg'].copy()
br_reg['name'] = br_reg['name'].astype(str).str.replace("_PV", "_BR", regex=False)
br_reg['desc'] = 'Поломки'
br_reg['scale_min'] = 0
br_reg['scale_max'] = 6
br_reg['dec'] = np.nan
br_reg['lolo'] = np.nan
br_reg['lo'] = np.nan
br_reg['hi'] = np.nan
br_reg['hihi'] = np.nan
br_reg['unit'] = np.nan
br_reg['logged'] = 0
brc_reg = table_result[table_result["type"] == 'reg'].copy()
brc_reg['name'] = brc_reg['name'].astype(str).str.replace("_PV", "_BRC", regex=False)
brc_reg['desc'] = 'Поломки'
brc_reg['scale_min'] = 0
brc_reg['scale_max'] = 6
brc_reg['dec'] = np.nan
brc_reg['lolo'] = np.nan
brc_reg['lo'] = np.nan
brc_reg['hi'] = np.nan
brc_reg['hihi'] = np.nan
brc_reg['unit'] = np.nan
brc_reg['logged'] = 0
br_reg = pd.concat([br_reg, brc_reg], ignore_index=True)
br_reg.head(2)

,name,tag_type,initial_value,desc,unit,scale_min,scale_max,dec,access_type,group_name,logged,lo,lolo,hi,hihi,discrete_alarm_value,type,name_copy
0,FIC3028_BR,1,0,Поломки,NaN,0,6,NaN,1,<NA>,0,NaN,NaN,NaN,NaN,<NA>,reg,FIC 3028
1,FIC3135_BR,1,0,Поломки,NaN,0,6,NaN,1,<NA>,0,NaN,NaN,NaN,NaN,<NA>,reg,FIC 3135


In [19]:
#sv
sv_reg = table_result[table_result["type"] == 'reg'].copy()
sv_reg['name'] = sv_reg['name'].astype(str).str.replace("_PV", "_SV", regex=False)
sv_reg['logged'] = 1
sv_reg.head(2)

,name,tag_type,initial_value,desc,unit,scale_min,scale_max,dec,access_type,group_name,logged,lo,lolo,hi,hihi,discrete_alarm_value,type,name_copy
72,FIC3028_SV,1,0,Расход. Водяной пар. Тр-д П1/2,кг/ч,15.0,500.0,<NA>,1,<NA>,1,NaN,NaN,NaN,NaN,<NA>,reg,FIC 3028
73,FIC3135_SV,1,0,Расход. Сероводородсодержащий газ. Тр-д 405/5,нм3/ч,55.0,600.0,<NA>,1,<NA>,1,NaN,NaN,NaN,NaN,<NA>,reg,FIC 3135


In [20]:
mv_reg = table_result[table_result["type"] == 'reg'].copy()
mv_reg['name'] = mv_reg['name'].astype(str).str.replace("_PV", "_MV", regex=False)
mv_reg['scale_min'] = 0
mv_reg['scale_max'] = 100
mv_reg['dec'] = 1
mv_reg['lolo'] = np.nan
mv_reg['lo'] = np.nan
mv_reg['hi'] = np.nan
mv_reg['hihi'] = np.nan
mv_reg['unit'] = '%'
mv_reg['logged'] = 0
mv_reg.head(2)

,name,tag_type,initial_value,desc,unit,scale_min,scale_max,dec,access_type,group_name,logged,lo,lolo,hi,hihi,discrete_alarm_value,type,name_copy
72,FIC3028_MV,1,0,Расход. Водяной пар. Тр-д П1/2,%,0,100,1,1,<NA>,0,NaN,NaN,NaN,NaN,<NA>,reg,FIC 3028
73,FIC3135_MV,1,0,Расход. Сероводородсодержащий газ. Тр-д 405/5,%,0,100,1,1,<NA>,0,NaN,NaN,NaN,NaN,<NA>,reg,FIC 3135


In [21]:
mode_reg = table_result[table_result["type"] == 'reg'].copy()
mode_reg['name'] = mode_reg['name'].astype(str).str.replace("_PV", "_MODE", regex=False)
mode_reg['scale_min'] = 0
mode_reg['scale_max'] = 3
mode_reg['dec'] = np.nan
mode_reg['lolo'] = np.nan
mode_reg['lo'] = np.nan
mode_reg['hi'] = np.nan
mode_reg['hihi'] = np.nan
mode_reg['unit'] = np.nan
mode_reg['logged'] = 0
mode_reg.head(2)

,name,tag_type,initial_value,desc,unit,scale_min,scale_max,dec,access_type,group_name,logged,lo,lolo,hi,hihi,discrete_alarm_value,type,name_copy
72,FIC3028_MODE,1,0,Расход. Водяной пар. Тр-д П1/2,NaN,0,3,NaN,1,<NA>,0,NaN,NaN,NaN,NaN,<NA>,reg,FIC 3028
73,FIC3135_MODE,1,0,Расход. Сероводородсодержащий газ. Тр-д 405/5,NaN,0,3,NaN,1,<NA>,0,NaN,NaN,NaN,NaN,<NA>,reg,FIC 3135


In [22]:
reg = pd.concat([mv_reg, sv_reg], ignore_index=True)
reg = pd.concat([reg, mode_reg], ignore_index=True)
reg.head(2)

C:\Users\Serenko\AppData\Local\Temp\ipykernel_16720\3268537467.py:2: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  reg = pd.concat([reg, mode_reg], ignore_index=True)


,name,tag_type,initial_value,desc,unit,scale_min,scale_max,dec,access_type,group_name,logged,lo,lolo,hi,hihi,discrete_alarm_value,type,name_copy
0,FIC3028_MV,1,0,Расход. Водяной пар. Тр-д П1/2,%,0.0,100.0,1,1,<NA>,0,NaN,NaN,NaN,NaN,<NA>,reg,FIC 3028
1,FIC3135_MV,1,0,Расход. Сероводородсодержащий газ. Тр-д 405/5,%,0.0,100.0,1,1,<NA>,0,NaN,NaN,NaN,NaN,<NA>,reg,FIC 3135


In [23]:
##apparats
mv_app = table_result[table_result["type"] == 'app'].copy()
mv_app['name'] = mv_app['name'].astype(str).str.replace("_PV", "_MV", regex=False)
mv_app['scale_min'] = 0
mv_app['scale_max'] = 1
mv_app['dec'] = np.nan
mv_app['lolo'] = np.nan
mv_app['lo'] = np.nan
mv_app['hi'] = np.nan
mv_app['hihi'] = np.nan
mv_app['unit'] = np.nan
mv_app['logged'] = 0
mv_app.head(2)

,name,tag_type,initial_value,desc,unit,scale_min,scale_max,dec,access_type,group_name,logged,lo,lolo,hi,hihi,discrete_alarm_value,type,name_copy
1,K_412_6_MV,1,0,Электрообогрев 412/6 отключить,NaN,0,1,NaN,1,<NA>,0,NaN,NaN,NaN,NaN,<NA>,app,412/6
2,AV_1_MV,1,0,Пуск вентиляции,NaN,0,1,NaN,1,<NA>,0,NaN,NaN,NaN,NaN,<NA>,app,АВ-1


In [24]:
#reg.to_csv("result.csv", index=False, encoding="utf-8")

In [25]:
#res.dtypes

In [26]:
#res = pd.concat([table_result, br_reg, reg], ignore_index=True)
#res